# BeautifulSoup Extraction Patterns

This notebook explains the scraping logic behind the Day 2 scripts.

The core workflow is:

1. start with HTML text
2. parse it into a `soup` object
3. identify repeated record containers
4. select fields inside each record
5. extract visible text or attributes
6. store one row per record
7. convert rows into a DataFrame

This is the same logic used in the static scraping walkthrough, but here the HTML is small enough to inspect by eye.


## 1. Imports

We use two packages:

- `BeautifulSoup` from `bs4` parses HTML and lets us search it.
- `pandas` turns extracted rows into tables.


In [3]:
# urljoin turns relative links such as /course/... into full URLs.
from urllib.parse import urljoin

# BeautifulSoup is the HTML parser/search tool.
from bs4 import BeautifulSoup

# pandas is used after extraction, when we want a data table.
import pandas as pd

# for pretty prints
from pprint import pprint

# HTML and display let the notebook render a small HTML fragment visually.
from IPython.display import HTML, display


## 3. Practice HTML for Selector Patterns

The artificial page below mimics a small part of a news outlet's search or topic page. It is not copied from a real outlet, but it uses common patterns from news sites.

The page contains two repeated article teasers. Each teaser is an `<article>` with:

- a headline link: `<h2 class="headline"><a href=...>`
- a short standfirst/summary paragraph: `<p class="summary">`
- a byline: `<span class="byline">`
- a publication date: `<time class="published" datetime=...>`
- a section label, such as `Politics` or `Technology`
- topic/tag links inside `<ul class="tags">`
- attributes such as `class`, `data-id`, `data-kind`, and `aria-label`

This is a small version of the kind of structure you might see on a news homepage, search-results page, or topic page.

In [ ]:
# This is raw HTML stored as a Python string.
# In a real scraper, similar HTML usually comes from response.text.
html = """
<html>
  <body>
    <header>
      <a class="site-logo" href="/">Example Daily</a>
    </header>

    <main id="results" data-page="search-results">
      <article class="result card featured" data-id="news-001" data-kind="result-article" aria-label="featured result">
        <p class="section-label">Politics</p>
        <h2 class="headline"><a href="/news/2026/dsa-election-platforms">Platforms face new election transparency questions</a></h2>
        <p class="summary">Regulators and researchers are watching how large platforms report election-related moderation decisions.</p>
        <p class="meta">
          <span class="byline">By Noelle Example</span>
          <time class="published" datetime="2026-06-25">25 June 2026</time>
        </p>
        <ul class="tags">
          <li><a class="tag" href="/topics/dsa">DSA</a></li>
          <li><a class="tag" href="/topics/elections">elections</a></li>
        </ul>
      </article>

      <article class="result card" data-id="news-002" data-kind="result-article" aria-label="ordinary result">
        <p class="section-label">Technology</p>
        <h2 class="headline"><a href="/news/2026/research-api-access">Researchers test platform API access routes</a></h2>
        <p class="summary">A new project compares API data, scraped public pages, and transparency database records.</p>
        <p class="meta">
          <span class="byline">By Alex Researcher</span>
          <time class="published" datetime="2026-06-24">24 June 2026</time>
        </p>
        <ul class="tags">
          <li><a class="tag" href="/topics/apis">APIs</a></li>
          <li><a class="tag" href="/topics/scraping">scraping</a></li>
        </ul>
      </article>
    </main>

    <aside data-note="not-a-result">This mentions result but is not a news article teaser.</aside>
    <a class="newsletter-signup">Newsletter sign-up link without href</a>
  </body>
</html>
"""

print(type(html))

<class 'str'>

<html>
  <body>
    <header>
      <a class="site-logo" href="/">Example Daily</a>
    </header>

    <main id="results" data-page="search-results">
      <article class="result card featured" data-id="news-001" data-kind="result-article" aria-label="featured result">
        <p class="section-labe


At this point, `html` is just text. Python does not yet understand the tags, links, classes, or attributes. BeautifulSoup creates a searchable representation of that HTML.


In [10]:
# BeautifulSoup turns HTML text into a searchable parse tree.
# "html.parser" is Python's built-in HTML parser.
soup = BeautifulSoup(html, "html.parser")

# prettify() is for inspection: it prints the nested HTML with indentation.
print(soup.prettify()[:1000])


<html>
 <body>
  <header>
   <a class="site-logo" href="/">
    Example Daily
   </a>
  </header>
  <main data-page="search-results" id="results">
   <article aria-label="featured result" class="result card featured" data-id="news-001" data-kind="result-article">
    <p class="section-label">
     Politics
    </p>
    <h2 class="headline">
     <a href="/news/2026/dsa-election-platforms">
      Platforms face new election transparency questions
     </a>
    </h2>
    <p class="summary">
     Regulators and researchers are watching how large platforms report election-related moderation decisions.
    </p>
    <p class="meta">
     <span class="byline">
      By Noelle Example
     </span>
     <time class="published" datetime="2026-06-25">
      25 June 2026
     </time>
    </p>
    <ul class="tags">
     <li>
      <a class="tag" href="/topics/dsa">
       DSA
      </a>
     </li>
     <li>
      <a class="tag" href="/topics/elections">
       elections
      </a>
     </li>
    </

## 3. HTML Elements: Tag, Text, Attribute

An HTML element can contain:

- a tag name, such as `article`, `a`, `span`, `time`, or `p`
- visible text, such as `Platforms face new election transparency questions`
- attributes, such as `class`, `href`, `datetime`, or `data-id`

Example: `<time class="published" datetime="2026-06-25">25 June 2026</time>`

- tag name: `time`
- visible text: `25 June 2026`
- class attribute: `published`
- datetime attribute: `2026-06-25`

This distinction matters because scraping often combines visible text and hidden attributes. For example, the human-readable date may be `25 June 2026`, while the machine-readable date is stored in `datetime="2026-06-25"`.

In [13]:
# select_one("a") returns the first <a> element.
first_link = soup.select_one("a")

print("Whole element:", first_link)
print("Tag name:", first_link.name)

# get_text(strip=True) extracts visible text and removes surrounding whitespace.
print("Visible text:", first_link.get_text(strip=True))

# .get("href") extracts an attribute. If it is missing, .get() returns None.
print("href attribute:", first_link.get("href"))
print("class attribute:", first_link.get("class"))


Whole element: <a class="site-logo" href="/">Example Daily</a>
Tag name: a
Visible text: Example Daily
href attribute: /
class attribute: ['site-logo']


## 4. What Selectors Are

A selector tells BeautifulSoup which HTML elements to find.

Common and useful selector forms:

| Selector | Meaning |
|---|---|
| `article` | all `<article>` elements |
| `a` | all `<a>` link elements, whether or not they have an `href` |
| `a[href]` | all `<a>` links that have an `href` attribute |
| `.result` | all elements with class `result` |
| `.summary` | all elements with class `summary` |
| `#results` | the element with id `results` |
| `.result.card` | one element that has both classes `result` and `card` |
| `.tags a.tag` | `<a>` links with class `tag` inside an element with class `tags` |
| `[data-id]` | all elements that have a `data-id` attribute |
| `[data-id="news-001"]` | elements whose `data-id` is exactly `news-001` |
| `[data-kind*="result"]` | elements whose `data-kind` attribute contains `result` |
| `[href^="/news/"]` | links whose `href` starts with `/news/` |
| `[href$="apis"]` | links whose `href` ends with `apis` |
| `#results > article` | `<article>` elements that are direct children of `#results` |
| `#results a` | all links anywhere inside `#results` |
| `article:first-of-type` | the first `<article>` among its siblings |
| `article:nth-of-type(2)` | the second `<article>` among its siblings |
| `article:not(.featured)` | `<article>` elements that do not have class `featured` |
| `.summary, .byline` | elements matching either `.summary` or `.byline` |

The dot in `.result` means class. The hash in `#results` means id. Square brackets such as `[data-id]` select by attributes.

Important caveat: CSS selectors usually select a **specific attribute**. For example, `[data-kind*="result"]` means "the `data-kind` attribute contains `result`." If you want to search **any attribute value** for a word such as `result`, use BeautifulSoup/Python logic rather than a CSS selector. An example appears below.

For more selector examples, see:

- MDN CSS Selectors: https://developer.mozilla.org/en-US/docs/Web/CSS/CSS_selectors
- MDN Selectors and Combinators: https://developer.mozilla.org/en-US/docs/Web/CSS/CSS_selectors/Selectors_and_combinators
- BeautifulSoup CSS selectors: https://www.crummy.com/software/BeautifulSoup/bs4/doc/#css-selectors

BeautifulSoup uses CSS-style selectors in `soup.select()` and `soup.select_one()`.
The MDN pages are useful for learning the general selector syntax, while the
BeautifulSoup documentation shows how selectors work inside Python. Always test
selectors on the actual saved HTML before using them at scale.

## 4b. When CSS Selectors Are Not Enough: Any Attribute Value

CSS attribute selectors are excellent when you know the attribute name: `[data-kind*="result"]`, `[href^="/posts/"]`, `[data-id="p1"]`.

But sometimes you want a broader diagnostic question: "Which elements have **any attribute value** containing the word `result`?" That is not a normal CSS selector pattern. For that, use Python logic with BeautifulSoup.


In [14]:
def any_attribute_value_contains(element, needle):
    # element.attrs is a dictionary of all attributes on an HTML element.
    # Example: {"class": ["result", "card"], "data-id": "p1"}
    for value in element.attrs.values():
        # BeautifulSoup stores class values as lists, e.g. ["result", "card"].
        # Other attributes, such as href or data-id, are usually strings.
        if isinstance(value, list):
            value_as_text = " ".join(value)
        else:
            value_as_text = str(value)

        if needle in value_as_text:
            return True

    return False


# find_all(...) can take a function. BeautifulSoup calls the function on each
# element and keeps the elements where the function returns True.
elements_with_result_in_any_attribute = soup.find_all(
    lambda element: any_attribute_value_contains(element, "result")
)

for element in elements_with_result_in_any_attribute:
    print(describe_element(element))

{'tag': 'main', 'id': 'results', 'class': '', 'href': None, 'data-id': None, 'data-kind': None, 'datetime': None, 'text': 'Politics Platforms face new election transparency questions Regulat...'}
{'tag': 'article', 'id': None, 'class': 'result card featured', 'href': None, 'data-id': 'news-001', 'data-kind': 'result-article', 'datetime': None, 'text': 'Politics Platforms face new election transparency questions Regulat...'}
{'tag': 'article', 'id': None, 'class': 'result card', 'href': None, 'data-id': 'news-002', 'data-kind': 'result-article', 'datetime': None, 'text': 'Technology Researchers test platform API access routes A new projec...'}
{'tag': 'aside', 'id': None, 'class': '', 'href': None, 'data-id': None, 'data-kind': None, 'datetime': None, 'text': 'This mentions result but is not a news article teaser.'}


## 5. `select_one()` vs `select()`

- `select_one(selector)` returns the first matching element, or `None`.
- `select(selector)` returns a list of all matching elements.

Use `select_one()` when you expect one value. Use `select()` when you expect several values.


In [ ]:
# First matching element with class="result".
first_tag = soup.select_one(".result")
print(first_tag)
print(first_tag.get("data-id"))

<article aria-label="featured result" class="result card featured" data-id="news-001" data-kind="result-article">
<p class="section-label">Politics</p>
<h2 class="headline"><a href="/news/2026/dsa-election-platforms">Platforms face new election transparency questions</a></h2>
<p class="summary">Regulators and researchers are watching how large platforms report election-related moderation decisions.</p>
<p class="meta">
<span class="byline">By Noelle Example</span>
<time class="published" datetime="2026-06-25">25 June 2026</time>
</p>
<ul class="tags">
<li><a class="tag" href="/topics/dsa">DSA</a></li>
<li><a class="tag" href="/topics/elections">elections</a></li>
</ul>
</article>
news-001


In [20]:
# All matching elements with class="result".
all_tags = soup.select(".result")
print(len(all_tags))
for article_tag in all_tags:
    print(article_tag.get("data-id"))

2
news-001
news-002


## 6. Multiple Classes: `.result.card` vs `.result .card`

No space: `.result.card`

Means: one element that has both classes `result` and `card`.

With a space: `.result .card`

Means: an element with class `card` somewhere inside an element with class `result`.

Those are different structural claims about the page. In the fake news HTML, each article teaser itself has both classes, so `.result.card` works. There is no separate nested `.card` element inside `.result`, so `.result .card` returns nothing.

In [16]:
# No space: elements that have both classes at the same time.
for card in soup.select(".result.card"):
    print(card.get("data-id"), card.get("class"))


news-001 ['result', 'card', 'featured']
news-002 ['result', 'card']


In [17]:
# With a space: .card elements nested inside .result elements.
# Our example has no nested .card elements, so this returns an empty list.
print(soup.select(".result .card"))


[]


## 7. Select Inside One Record

A central scraping rule:

First select the repeated record container, then select fields inside that container.

This keeps the title, summary, author, and tags from the same record together.


In [ ]:
# One repeated record container.
card = soup.select_one(".result.card")

# These selectors are searched inside this one news card, not across the whole page.
headline_link = card.select_one(".headline a")
summary = card.select_one(".summary")
byline = card.select_one(".byline")
published = card.select_one("time.published")
section = card.select_one(".section-label")
tag_links = card.select(".tags a.tag")

print("headline element:", headline_link)
print("summary element:", summary)
print("byline element:", byline)
print("published date element:", published)
print("section element:", section)
print("tag elements:", tag_links)

## 8. Extract Visible Text

`get_text(" ", strip=True)` is used often in the scripts.

It means:

- extract visible text
- join nested text with spaces
- remove leading and trailing whitespace

This matters because HTML often contains line breaks and indentation that should not become data.


In [ ]:
# Extract visible text from selected elements.
headline_text = headline_link.get_text(" ", strip=True)
summary_text = summary.get_text(" ", strip=True)
byline_text = byline.get_text(" ", strip=True)
section_text = section.get_text(" ", strip=True)
published_text = published.get_text(" ", strip=True)

print(headline_text)
print(summary_text)
print(byline_text)
print(section_text)
print(published_text)

## 9. Extract Attributes

Some useful data is not visible text.

For example:

- the article URL is in the `href` attribute of the headline link
- the article ID is in the `data-id` attribute of the article card
- the machine-readable publication date is in the `datetime` attribute
- class labels are in the `class` attribute

Use `.get("attribute_name")` for attributes.

In [ ]:
# data-id is an attribute on the article element.
article_id = card.get("data-id")

# href is an attribute on the headline link.
article_url = headline_link.get("href")

# datetime is an attribute on the <time> element. It is often cleaner for data
# analysis than the visible date string because it follows a regular format.
published_date = published.get("datetime")

# class is also an attribute; BeautifulSoup returns class labels as a list.
classes = card.get("class")

print(article_id)
print(article_url)
print(published_date)
print(classes)

## 10. Defensive Extraction

Real pages are inconsistent. One card may be missing an author, date, image, or link.

This pattern prevents a crash and stores missingness explicitly:

`value = element.get_text(" ", strip=True) if element else None`


In [ ]:
# There is no image element in this example.
image_element = card.select_one("img")
print(image_element)

# Safe extraction: store None if the element is missing.
image_url = image_element.get("src") if image_element else None
print(image_url)

## 11. One Record Becomes One Row

A row is usually a dictionary. The dictionary keys become column names.


In [ ]:
row = {
    "article_id": card.get("data-id"),
    "headline": headline_link.get_text(" ", strip=True) if headline_link else None,
    "article_url": headline_link.get("href") if headline_link else None,
    "summary": summary.get_text(" ", strip=True) if summary else None,
    "byline": byline.get_text(" ", strip=True) if byline else None,
    "section": section.get_text(" ", strip=True) if section else None,
    "published_text": published.get_text(" ", strip=True) if published else None,
    "published_date": published.get("datetime") if published else None,
    "topics": [tag.get_text(" ", strip=True) for tag in tag_links],
    "topic_count": len(tag_links),
}

row

## 12. Many Records Become a DataFrame

Now repeat the same extraction for every record container.


In [ ]:
rows = []

# .result.card selects the repeated news article teaser containers.
for card in soup.select(".result.card"):
    # Search inside the current card to keep fields from the same article together.
    headline_link = card.select_one(".headline a")
    summary = card.select_one(".summary")
    byline = card.select_one(".byline")
    published = card.select_one("time.published")
    section = card.select_one(".section-label")
    tag_links = card.select(".tags a.tag")

    row = {
        "article_id": card.get("data-id"),
        "headline": headline_link.get_text(" ", strip=True) if headline_link else None,
        "article_url": headline_link.get("href") if headline_link else None,
        "summary": summary.get_text(" ", strip=True) if summary else None,
        "byline": byline.get_text(" ", strip=True) if byline else None,
        "section": section.get_text(" ", strip=True) if section else None,
        "published_text": published.get_text(" ", strip=True) if published else None,
        "published_date": published.get("datetime") if published else None,
        "topics": [tag.get_text(" ", strip=True) for tag in tag_links],
        "topic_count": len(tag_links),
        # Derived Boolean field: True if the card has class="featured".
        "is_featured": "featured" in card.get("class", []),
    }

    rows.append(row)

# pd.DataFrame turns the list of row dictionaries into a table.
df = pd.DataFrame(rows)
df

## 13. Long Table for Repeated Values

The `topics` field has multiple values. Often it is better to create a separate table with one row per article-topic pair.

This is useful when you want to count topics, join topic metadata, or avoid storing lists inside a CSV cell.

In [ ]:
topic_rows = []

for card in soup.select(".result.card"):
    article_id = card.get("data-id")
    headline_link = card.select_one(".headline a")
    headline = headline_link.get_text(" ", strip=True) if headline_link else None

    for tag in card.select(".tags a.tag"):
        topic_rows.append(
            {
                "article_id": article_id,
                "headline": headline,
                "topic": tag.get_text(" ", strip=True),
                "topic_url": tag.get("href"),
            }
        )

topics_df = pd.DataFrame(topic_rows)
topics_df

## 14. Direct Children vs Descendants

A space means descendant at any depth: `#results a` means all links anywhere inside `#results`.

The `>` symbol means direct child: `#results > article` means articles immediately inside `#results`.


In [ ]:
# Descendant selector: all links anywhere inside #results.
all_links_inside_results = soup.select("#results a")
print([link.get_text(" ", strip=True) for link in all_links_inside_results])


In [ ]:
# Direct-child selector: only article elements immediately inside #results.
direct_articles = soup.select("#results > article")
print([article.get("data-id") for article in direct_articles])


## 15. Real-Page Fragment: MethodsNET Course List

The fake news HTML above is useful because it is small and clean. Real websites are usually messier.

The next example uses a **shortened teaching excerpt** based on the public MethodsNET course list. We do not paste the whole website into the notebook. Instead, we keep only the part needed to teach the extraction pattern: a table with repeated course rows.

This shows a slightly different situation:

- the repeated record container is a table row: `tbody tr`;
- fields are stored in table cells: `td`;
- the course-detail URL is stored in an `href` attribute;
- some extraction depends on column position, which is common for tables but should be documented because it assumes the column order stays stable.

In [ ]:
# This is a shortened teaching excerpt based on the public MethodsNET course list.
# In the runnable workflow, similar HTML is fetched from:
# https://methodsnet.org/course-list/
methodsnet_course_list_url = "https://methodsnet.org/course-list/"

methodsnet_list_html = """
<table class="course-list">
  <thead>
    <tr>
      <th>Full Details</th>
      <th>Code</th>
      <th>Courses</th>
      <th>Instructor name</th>
      <th>Week</th>
      <th>Confirmation</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td><a href="/course/d04-collecting-data-from-large-online-platforms-with-apis-webscraping-and-dsa-applications/">Click for details</a></td>
      <td>D04</td>
      <td>Collecting Data from Large Online Platforms with APIs, Webscraping, and DSA applications</td>
      <td>Noëlle Lebernegg</td>
      <td>1</td>
      <td>Confirmed</td>
    </tr>
    <tr>
      <td><a href="/course/b01-critical-discourse-analysis/">Click for details</a></td>
      <td>B01</td>
      <td>Critical Discourse Analysis: Texts, Contexts and Power</td>
      <td>Sam Bennett</td>
      <td>1</td>
      <td>Confirmed</td>
    </tr>
  </tbody>
</table>
"""

# Parse the HTML excerpt just as we parsed the fake news example.
methodsnet_soup = BeautifulSoup(methodsnet_list_html, "html.parser")

# tbody tr selects all table rows inside the table body. Here, each row is one
# course record. This is the table equivalent of selecting .result.card above.
course_table_rows = methodsnet_soup.select("tbody tr")
print("Number of course rows:", len(course_table_rows))
print(course_table_rows[0].prettify())

## 16. Extract Rows from the MethodsNET Table Fragment

For table-like pages, a common pattern is:

1. select the repeated rows;
2. select the cells inside each row;
3. assign cells to variables based on the table header;
4. extract attributes such as `href` from links;
5. turn the rows into a DataFrame.

This is not “better” or “worse” than class-based selectors. It is just a different page structure. The methodological point is the same: selectors define what counts as a record and what becomes a field.

In [ ]:
methodsnet_rows = []

for table_row in methodsnet_soup.select("tbody tr"):
    # td selects all table cells in the current row.
    cells = table_row.select("td")

    # The detail link is in the first cell. a[href] means: find a link element
    # that has an href attribute.
    detail_link = table_row.select_one("td a[href]")

    # Defensive check: if the row does not have the expected number of cells,
    # skip it or handle it separately. Real websites often contain header rows,
    # empty rows, or advertisement rows.
    if len(cells) < 6 or detail_link is None:
        continue

    methodsnet_rows.append(
        {
            # urljoin() turns the relative URL /course/... into a full URL.
            "course_url": urljoin(methodsnet_course_list_url, detail_link.get("href")),
            # The following fields come from visible text in specific table cells.
            # This assumes the column order follows the header above.
            "course_code": cells[1].get_text(" ", strip=True),
            "course_title": cells[2].get_text(" ", strip=True),
            "instructor": cells[3].get_text(" ", strip=True),
            "week": cells[4].get_text(" ", strip=True),
            "status": cells[5].get_text(" ", strip=True),
        }
    )

methodsnet_df = pd.DataFrame(methodsnet_rows)
methodsnet_df

## 17. Real-Page Fragment: MethodsNET Course Detail Page

A course-detail page has a different structure from the course list. Instead of a table, it often contains headings and text blocks.

The small excerpt below illustrates another common scraping task: find a heading such as `Date & Time` or `Instructor`, then collect the nearby value. This is more fragile than a clean table, so it is especially important to save raw HTML and document the parsing rule.

In [ ]:
methodsnet_detail_html = """
<article class="course-detail" data-course-code="D04">
  <h1>D04: Collecting Data from Large Online Platforms with APIs, Webscraping and DSA applications</h1>

  <section class="course-facts">
    <h3>Date &amp; Time</h3>
    <p>29.06.26 - 03.07.26</p>

    <h3>Course Time</h3>
    <p>09:00-12:00 &amp; 13:30-15:15</p>

    <h3>Instructor</h3>
    <p>Noëlle Sophie Lebernegg</p>

    <h3>ECTS</h3>
    <p>4</p>
  </section>

  <section class="description">
    <h2>Short Description</h2>
    <p>This course introduces advanced techniques for collecting and managing data from VLOPs.</p>

    <h2>Learning Objectives</h2>
    <ul>
      <li>Collect data using open APIs.</li>
      <li>Build ethical and robust web scrapers.</li>
      <li>Document reproducible data collection workflows.</li>
    </ul>
  </section>
</article>
"""

methodsnet_detail_soup = BeautifulSoup(methodsnet_detail_html, "html.parser")

# The h1 is usually the most direct place to get the course title.
course_title = methodsnet_detail_soup.select_one("h1").get_text(" ", strip=True)

# data-course-code is an attribute on the article container. Attribute values
# are often useful IDs because they are less ambiguous than visible titles.
course_code = methodsnet_detail_soup.select_one("article.course-detail").get("data-course-code")

print(course_code)
print(course_title)

## 18. Extract Values Near Headings

The function below is a small parser for the course-detail excerpt.

It searches for a heading with a given label, such as `Instructor`, and then takes the next paragraph as the value. This is a useful pattern, but it is also an assumption about page structure. If the website changes from paragraphs to tables or cards, this function may need to change.

In [ ]:
def value_after_heading(soup, heading_text):
    # find(...) can take a function. BeautifulSoup tests each element and keeps
    # the first one where the function returns True.
    heading = soup.find(
        lambda element: element.name in ["h2", "h3", "h4"]
        and element.get_text(" ", strip=True).lower() == heading_text.lower()
    )

    if heading is None:
        return None

    # find_next("p") finds the next paragraph after the heading. In this excerpt,
    # each fact heading is followed by one paragraph containing the value.
    value_paragraph = heading.find_next("p")
    return value_paragraph.get_text(" ", strip=True) if value_paragraph else None


course_detail_row = {
    "course_code": course_code,
    "course_title": course_title,
    "date_time": value_after_heading(methodsnet_detail_soup, "Date & Time"),
    "course_time": value_after_heading(methodsnet_detail_soup, "Course Time"),
    "instructor": value_after_heading(methodsnet_detail_soup, "Instructor"),
    "ects": value_after_heading(methodsnet_detail_soup, "ECTS"),
    "short_description": value_after_heading(methodsnet_detail_soup, "Short Description"),
}

course_detail_row

## 19. Extract Repeated List Items from the Course Detail Page

The learning objectives are stored as repeated `<li>` elements inside a section. This is the same repeated-value problem as article topics above: one course can have several objectives.

Depending on the research question, you could store these as a list in one row, or create a long table with one row per course-objective pair.

In [ ]:
objective_items = methodsnet_detail_soup.select(".description li")

objectives = [item.get_text(" ", strip=True) for item in objective_items]
print(objectives)

objective_rows = []
for number, objective in enumerate(objectives, start=1):
    objective_rows.append(
        {
            "course_code": course_code,
            "objective_number": number,
            "objective": objective,
        }
    )

objectives_df = pd.DataFrame(objective_rows)
objectives_df

## 20. How This Maps to the Static Scraping Walkthrough

In the static scraping walkthrough, the page is `https://quotes.toscrape.com/`.

There, the repeated container is `.quote`.

Inside each `.quote`, the script looks for:

- `.text`: quote text
- `.author`: author name
- `.tags a.tag`: tag links
- `span a[href]`: author profile link

The page is different, but the workflow is the same:

1. find repeated record containers
2. inspect one record
3. select fields inside one record
4. loop over records
5. build rows
6. create a DataFrame


## From Patterns to Workflow

This notebook focuses on selector mechanics: how to read HTML, choose selectors,
extract text and attributes, and turn repeated elements into rows.

The full scraping workflow, including real-page collection, raw HTML storage,
link extraction, pagination, and output files, is in
`day_2_static_scraping_walkthrough.ipynb`.

The MethodsNET-specific runnable workflow is in
`scripts/runnable_workflows/03b_methodsnet_course_scraper.py`. It uses the same
ideas shown in the small MethodsNET fragments here, but packages them into a
reproducible script that fetches the public course list, saves raw HTML, extracts
course links, and optionally follows detail pages.

## 21. Mini Exercise

Using the fake news HTML in this notebook:

1. Explain what `.result.card` selects.
2. Explain why `.result .card` returns nothing here.
3. Add a column called `n_words_summary` that counts words in each summary.
4. Create a long table with one row per article-topic combination.
5. Explain which fields come from visible text and which fields come from attributes.
6. Extract the machine-readable publication date from the `datetime` attribute.

Using the MethodsNET excerpt:

7. Explain why `tbody tr` is the repeated record selector.
8. Explain why extracting `cells[2]` assumes a stable table column order.
9. Add a column that combines course code and title.
10. Create a long table with one row per learning objective.

## Key Takeaway

BeautifulSoup does not decide what counts as a record or field. You decide that through selectors.

That means selectors are methodological choices, not just technical details.